# 14 - Treinar e Serializar o Modelo Final (Fase 7 - Refatoracao)

Usa exclusivamente `src/` (nenhuma logica reimplementada aqui). Treina os dois
finalistas com a configuracao congelada, serializa em `models/`, e verifica que o lucro
recalculado no teste bate EXATAMENTE com o notebook 12. Se nao bater, o notebook para -
nao ha "conserto" aqui, so verificacao.

In [1]:
import sys
import json
import hashlib
import platform
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost

sys.path.insert(0, str(Path('..').resolve()))

from src.data import load_split, FEATURE_SET, CATEGORICAL_COLS
from src.features import build_features, prepare_X
from src.economics import compute_interest_loss, profit_at_threshold
from src.models import build_xgb_final, build_logistic_pipeline

MODELS_DIR = Path('..') / 'models'
MODELS_DIR.mkdir(exist_ok=True)

THRESH_XGB = 0.31
THRESH_M1 = 0.38

print(f'sklearn {sklearn.__version__} | xgboost {xgboost.__version__} | python {platform.python_version()}')


sklearn 1.9.0 | xgboost 3.3.0 | python 3.12.10


## 1-2. Carregar train.parquet (ordem original) e aplicar build_features + FEATURE_SET

In [2]:
train_path = Path('..') / 'data' / 'processed' / 'train.parquet'
train = load_split('train')
print(f'train: {train.shape} (ordem original do arquivo, nao reordenada)')

train_feat = build_features(train)
X_train = prepare_X(train_feat, FEATURE_SET, CATEGORICAL_COLS)
y_train = train_feat['target'].values
print(f'X_train: {X_train.shape}')


train: (172988, 89) (ordem original do arquivo, nao reordenada)


X_train: (172988, 90)


## 3-4. Treinar XGB_walkforward (config congelada) e serializar

In [3]:
xgb_final = build_xgb_final()
xgb_final.fit(X_train, y_train)
print('XGB_walkforward treinado.')

joblib.dump(xgb_final, MODELS_DIR / 'xgb_final.joblib')
print('Serializado em models/xgb_final.joblib')


XGB_walkforward treinado.
Serializado em models/xgb_final.joblib


In [4]:
def sha256_of_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


train_hash = sha256_of_file(train_path)

model_meta = {
    'model': 'XGB_walkforward',
    'hiperparametros': xgb_final.get_params(),
    'FEATURE_SET': FEATURE_SET,
    'FEATURE_SET_n': len(FEATURE_SET),
    'threshold_operacional': THRESH_XGB,
    'data_treino_utc': datetime.now(timezone.utc).isoformat(),
    'sklearn_version': sklearn.__version__,
    'xgboost_version': xgboost.__version__,
    'python_version': platform.python_version(),
    'train_parquet_sha256': train_hash,
    'train_n_rows': int(len(train)),
    'X_train_columns': list(X_train.columns),
}

# get_params() pode conter objetos nao serializaveis (ex: None e tipos numpy) - forcar str
# em qualquer valor que nao seja tipo JSON nativo.
def _json_safe(v):
    if isinstance(v, (str, int, float, bool)) or v is None:
        return v
    return str(v)

model_meta['hiperparametros'] = {k: _json_safe(v) for k, v in model_meta['hiperparametros'].items()}

with open(MODELS_DIR / 'model_meta.json', 'w', encoding='utf-8') as f:
    json.dump(model_meta, f, indent=2, ensure_ascii=False)

print(f'models/model_meta.json salvo. SHA256 de train.parquet: {train_hash}')


models/model_meta.json salvo. SHA256 de train.parquet: 0f7e711edc7c20839c0bd569a0c6f6e3125cadd3dcd1264a57a323b2b89a1191


## 5. Verificacao de integridade - XGB_walkforward

Recarrega o `.joblib`, prediz no teste, calcula o lucro no threshold 0,31 e compara
EXATAMENTE com o notebook 12 ($242.230.710,89). Se divergir, o notebook para aqui.

In [5]:
REFERENCE_LUCRO_XGB = 242230710.89

xgb_reloaded = joblib.load(MODELS_DIR / 'xgb_final.joblib')

test = load_split('test')
test_feat = build_features(test)
X_test = prepare_X(test_feat, FEATURE_SET, CATEGORICAL_COLS)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
y_test = test_feat['target'].values

interest_test, loss_test = compute_interest_loss(test_feat)
interest_test = interest_test.values
loss_test = loss_test.values

y_prob_test_xgb = xgb_reloaded.predict_proba(X_test)[:, 1]
lucro_xgb_reproduzido = profit_at_threshold(y_test, y_prob_test_xgb, THRESH_XGB, interest_test, loss_test)

diff_xgb = lucro_xgb_reproduzido - REFERENCE_LUCRO_XGB
print(f'Lucro reproduzido (XGB_walkforward, modelo recarregado): $ {lucro_xgb_reproduzido:,.2f}')
print(f'Lucro de referencia (notebook 12):                       $ {REFERENCE_LUCRO_XGB:,.2f}')
print(f'Diferenca: $ {diff_xgb:,.4f}')

if abs(diff_xgb) > 0.01:
    raise RuntimeError(
        f'DIVERGENCIA: lucro reproduzido (${lucro_xgb_reproduzido:,.2f}) difere do notebook 12 '
        f'(${REFERENCE_LUCRO_XGB:,.2f}) em ${diff_xgb:,.2f}. Parando - nao ajustar, investigar a causa.'
    )
print('OK: bate exatamente (diferenca < $0,01) com o notebook 12.')


Lucro reproduzido (XGB_walkforward, modelo recarregado): $ 242,230,710.89
Lucro de referencia (notebook 12):                       $ 242,230,710.89
Diferenca: $ -0.0000
OK: bate exatamente (diferenca < $0,01) com o notebook 12.


## 6. Mesmo processo para M1 (LogisticRegression)

In [6]:
m1_pipeline = build_logistic_pipeline()
m1_pipeline.fit(X_train, y_train)
print('M1 treinado.')

joblib.dump(m1_pipeline, MODELS_DIR / 'logistic_baseline.joblib')
print('Serializado em models/logistic_baseline.joblib')


C:\Users\Avell\Documents\Projetos\credit-default-prediction-lendingclub\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


M1 treinado.
Serializado em models/logistic_baseline.joblib


In [7]:
REFERENCE_LUCRO_M1 = 235936408.63

m1_reloaded = joblib.load(MODELS_DIR / 'logistic_baseline.joblib')
y_prob_test_m1 = m1_reloaded.predict_proba(X_test)[:, 1]
lucro_m1_reproduzido = profit_at_threshold(y_test, y_prob_test_m1, THRESH_M1, interest_test, loss_test)

diff_m1 = lucro_m1_reproduzido - REFERENCE_LUCRO_M1
print(f'Lucro reproduzido (M1, modelo recarregado): $ {lucro_m1_reproduzido:,.2f}')
print(f'Lucro de referencia (notebook 12):          $ {REFERENCE_LUCRO_M1:,.2f}')
print(f'Diferenca: $ {diff_m1:,.4f}')

if abs(diff_m1) > 0.01:
    raise RuntimeError(
        f'DIVERGENCIA: lucro reproduzido (${lucro_m1_reproduzido:,.2f}) difere do notebook 12 '
        f'(${REFERENCE_LUCRO_M1:,.2f}) em ${diff_m1:,.2f}. Parando - nao ajustar, investigar a causa.'
    )
print('OK: bate exatamente (diferenca < $0,01) com o notebook 12.')


Lucro reproduzido (M1, modelo recarregado): $ 235,936,408.63
Lucro de referencia (notebook 12):          $ 235,936,408.63
Diferenca: $ 0.0000
OK: bate exatamente (diferenca < $0,01) com o notebook 12.


## Resumo

Ambos os modelos foram treinados via `src/models.py`, serializados em `models/`, e a
previsao do artefato recarregado reproduz exatamente o lucro reportado no notebook 12
para os dois. `models/model_meta.json` fica versionado; os `.joblib` ficam fora do git
(binarios grandes, reproduziveis a partir deste notebook + `src/`).